# Phase 10 — Classical baselines and dataset-of-origin test
* Data: features_v3 (the same final splits as SkyShield-IDS), all three configurations
* Baselines: XGBoost and Random Forest, with the same class weights w_c = N / (K * N_c) as SkyShield-IDS
  - XGBoost: hist trees, early stopping on the unweighted log loss of the complete validation split
  - Random Forest: 100 trees, each grown on a bootstrap sample of 500,000 training records (memory limit)
* Dataset-of-origin test (merged configuration only): can the 13 unified features tell which dataset a record
  comes from? XGBoost and decision trees of depth 1 and 2, trained on the merged training split, tested on the test split
* Metrics: Macro F1, accuracy, per-class precision/recall/F1/FPR, false alarm rate, confusion matrix, model size,
  CPU inference time; the merged models are also evaluated on each dataset separately, and on CIC-IoT2023
  without Spoofing (7 classes, a prediction of Spoofing counts as an error)
* Each run is saved as soon as it finishes; re-run Cell 2 after an interruption and finished runs are skipped
Outputs: outputs_v3/baselines_v1/ (results/, models/, baselines_v1_log.txt, summary.json)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings and helpers
import os, json, time, gc, platform, pickle
import numpy as np, pandas as pd
import xgboost as xgb, sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             balanced_accuracy_score, roc_auc_score)

FEAT = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
OUT  = '/content/drive/MyDrive/Thesis/outputs_v3/baselines_v1/'
for sub in ('results/', 'models/'):
    os.makedirs(OUT + sub, exist_ok=True)
SEED = 42
N_CLASSES = 8
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
CONFIGS = ['config1_merged', 'config2_cicids2017', 'config3_ciciot2023']
RUNS = ([('origin_test', 'origin', 'config1_merged')] + [(f'xgboost_{c}', 'xgboost', c) for c in CONFIGS]
        + [(f'random_forest_{c}', 'random_forest', c) for c in CONFIGS])
LOGFILE = OUT + 'baselines_v1_log.txt'
GPU = bool(os.popen('nvidia-smi -L 2>/dev/null').read().strip())

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:
        f.write(line + '\n')

def hm(sec):
    m = sec / 60
    return f'{int(m // 60)} h {int(round(m % 60)):02d} min' if m >= 60 else f'{m:.1f} min'

def save_atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def result_path(name):
    return OUT + f'results/{name}.json'

def load(cfg, split):
    X = np.load(FEAT + cfg + f'/X_{split}.npy').astype(np.float32, copy=False)
    y = np.load(FEAT + cfg + f'/y_{split}.npy').astype(np.int64)
    return X, y

def class_weights(y):
    """w_c = N / (K * N_c) over the K classes present in the training data (as for SkyShield-IDS)."""
    c = np.bincount(y, minlength=N_CLASSES); present = np.flatnonzero(c)
    return {int(k): float(len(y) / (len(present) * c[k])) for k in present}

def metrics(y, pred, labels=None):
    labels = np.unique(y) if labels is None else np.asarray(labels)
    pr, rc, f1c, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y, pred, labels=labels)
    tot = cm.sum(); fpr = {}
    for i, k in enumerate(labels):
        fp = cm[:, i].sum() - cm[i, i]; tn = tot - cm[i].sum() - fp
        fpr[CLASS_NAMES[k]] = float(fp / (fp + tn)) if fp + tn else 0.0
    far = None
    if 0 in labels:                                    # false alarm rate: benign records classified as any attack
        b = list(labels).index(0); far = float(1 - cm[b, b] / cm[b].sum())
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)),
                weighted_f1=float(f1_score(y, pred, average='weighted', labels=labels, zero_division=0)),
                false_alarm_rate=far, macro_fpr=float(np.mean(list(fpr.values()))),
                per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), fpr=fpr[CLASS_NAMES[k]],
                                                support=int(s_)) for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)},
                confusion_matrix=cm.tolist(), labels=[CLASS_NAMES[k] for k in labels], test_rows=int(len(y)),
                predicted_outside_labels=int((~np.isin(pred, labels)).sum()))

def log_metrics(prefix, r):
    log(f"{prefix}: Macro F1 {r['macro_f1']:.4f} | accuracy {r['accuracy']:.4f} | false alarm rate "
        f"{(r['false_alarm_rate'] or 0) * 100:.2f}% | {r['test_rows']:,} rows")
    log('   per class (P / R / F1 / FPR): ' + '; '.join(f"{k} {v['precision']:.3f}/{v['recall']:.3f}/{v['f1']:.3f}/{v['fpr']:.3f}"
                                                     for k, v in r['per_class'].items()))

def cpu_latency(predict_one, X, n=1024, warm=100):
    """Single-stream latency (one record per query), as for SkyShield-IDS: 100 warm-up queries, 1,024 timed queries."""
    rng = np.random.default_rng(SEED); idx = rng.integers(0, len(X), n + warm); ts = []
    for j, i in enumerate(idx):
        t = time.perf_counter(); predict_one(X[i:i + 1]); dt = time.perf_counter() - t
        if j >= warm: ts.append(dt * 1000)
    return dict(p90_ms=float(np.percentile(ts, 90)), mean_ms=float(np.mean(ts)))

def extra_merged(pred_fn, cfg, Xte, yte):
    """Merged configuration: results on the test rows of each dataset, and on CIC-IoT2023 without Spoofing."""
    out = {}
    if cfg != 'config1_merged':
        return out
    s = np.load(FEAT + cfg + '/source_test.npy')
    for src in ('CICIDS2017', 'CIC-IoT2023'):
        m = s == src; p = pred_fn(Xte[m])
        out[f'merged_test_{src}'] = metrics(yte[m], p)
        if src == 'CIC-IoT2023':
            k = yte[m] != 6
            out['merged_test_CIC-IoT2023_7classes'] = metrics(yte[m][k], p[k], labels=[c for c in range(N_CLASSES) if c != 6])
    return out

log('=' * 100)
log(f'Baselines v1 | Python {platform.python_version()} | xgboost {xgb.__version__} | scikit-learn {sklearn.__version__} | '
    f'GPU: {GPU} | CPU cores: {os.cpu_count()}')
log(f'{sum(os.path.exists(result_path(r[0])) for r in RUNS)} of {len(RUNS)} runs already finished')

In [ ]:
# %% Cell 2 — run all baselines (resumable)
T0 = time.time()
for i, (name, method, cfg) in enumerate(RUNS, 1):
    if os.path.exists(result_path(name)):
        log(f'run {i} of {len(RUNS)} ({name}): already finished, skipping'); continue
    log(f'--- run {i} of {len(RUNS)}: {name} ---')
    t0 = time.time()
    Xtr, ytr = load(cfg, 'train'); Xv, yv = load(cfg, 'val'); Xte, yte = load(cfg, 'test')
    log(f'{name}: training {len(ytr):,} rows, {Xtr.shape[1]} features | validation {len(yv):,} | test {len(yte):,} | '
        f'loaded in {time.time() - t0:.0f}s')
    res = dict(run=name, method=method, config=cfg, seed=SEED, train_rows=int(len(ytr)))

    if method == 'origin':
        s_tr = (np.load(FEAT + cfg + '/source_train.npy') == 'CIC-IoT2023').astype(np.int32)
        s_v = (np.load(FEAT + cfg + '/source_val.npy') == 'CIC-IoT2023').astype(np.int32)
        s_te = (np.load(FEAT + cfg + '/source_test.npy') == 'CIC-IoT2023').astype(np.int32)
        log(f'{name}: share of CIC-IoT2023 records — train {s_tr.mean():.3f}, test {s_te.mean():.3f}')
        clf = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6, tree_method='hist',
                                device='cuda' if GPU else 'cpu', early_stopping_rounds=20, eval_metric='logloss',
                                random_state=SEED, n_jobs=-1)
        clf.fit(Xtr, s_tr, eval_set=[(Xv, s_v)], verbose=False)
        prob = clf.predict_proba(Xte)[:, 1]; pred = (prob >= 0.5).astype(np.int32)
        feats = json.load(open(FEAT + cfg + '/spec.json'))['features']
        res['xgboost'] = dict(accuracy=float(accuracy_score(s_te, pred)), balanced_accuracy=float(balanced_accuracy_score(s_te, pred)),
                              auc=float(roc_auc_score(s_te, prob)), trees=int(clf.best_iteration + 1),
                              importance={f: float(v) for f, v in zip(feats, clf.feature_importances_)})
        log(f"{name}: XGBoost — accuracy {res['xgboost']['accuracy']:.5f} | balanced accuracy "
            f"{res['xgboost']['balanced_accuracy']:.5f} | AUC {res['xgboost']['auc']:.5f}")
        for depth in (1, 2):
            dt = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(Xtr, s_tr)
            p = dt.predict(Xte)
            res[f'tree_depth{depth}'] = dict(accuracy=float(accuracy_score(s_te, p)), balanced_accuracy=float(balanced_accuracy_score(s_te, p)),
                                             rules=export_text(dt, feature_names=feats))
            log(f"{name}: decision tree depth {depth} — accuracy {res[f'tree_depth{depth}']['accuracy']:.5f} | balanced accuracy "
                f"{res[f'tree_depth{depth}']['balanced_accuracy']:.5f}\n{res[f'tree_depth{depth}']['rules']}")
        # per class: does origin remain identifiable within each class?
        res['per_class_accuracy'] = {CLASS_NAMES[k]: float(accuracy_score(s_te[yte == k], pred[yte == k]))
                                     for k in np.unique(yte) if (s_te[yte == k].min() != s_te[yte == k].max())}
        log(f"{name}: XGBoost accuracy within each class present in both datasets: "
            + str({k: round(v, 4) for k, v in res['per_class_accuracy'].items()}))
    else:
        cw = class_weights(ytr); sw = np.vectorize(cw.get)(ytr).astype(np.float32)
        log(f'{name}: class weights ' + str({CLASS_NAMES[k]: round(v, 3) for k, v in cw.items()}))
        if method == 'xgboost':
            present = np.unique(ytr); remap = {int(k): j for j, k in enumerate(present)}; inv = np.array(present)
            ytr_m = np.vectorize(remap.get)(ytr); yv_m = np.vectorize(lambda k: remap.get(int(k), -1))(yv)
            keepv = yv_m >= 0
            clf = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.1, max_depth=8, tree_method='hist',
                                    device='cuda' if GPU else 'cpu', early_stopping_rounds=20, eval_metric='mlogloss',
                                    objective='multi:softprob', random_state=SEED, n_jobs=-1)
            t1 = time.time()
            clf.fit(Xtr, ytr_m, sample_weight=sw, eval_set=[(Xv[keepv], yv_m[keepv])], verbose=200)
            res['train_minutes'] = round((time.time() - t1) / 60, 2); res['trees'] = int(clf.best_iteration + 1)
            res['params'] = dict(n_estimators_max=2000, learning_rate=0.1, max_depth=8, early_stopping_rounds=20)
            pred_fn = lambda X: inv[clf.predict(X)]
            path = OUT + f'models/{name}.ubj'; clf.save_model(path)
            clf_cpu = xgb.XGBClassifier(); clf_cpu.load_model(path); clf_cpu.set_params(device='cpu', n_jobs=1)
            one = lambda x: clf_cpu.predict(x)
        else:
            clf = RandomForestClassifier(n_estimators=100, max_samples=min(500_000, len(ytr)), min_samples_leaf=1,
                                         class_weight=cw, n_jobs=-1, random_state=SEED)
            t1 = time.time(); clf.fit(Xtr, ytr)
            res['train_minutes'] = round((time.time() - t1) / 60, 2)
            res['params'] = dict(n_estimators=100, max_samples=min(500_000, len(ytr)), class_weight='N/(K*N_c)')
            res['nodes'] = int(sum(e.tree_.node_count for e in clf.estimators_))
            pred_fn = lambda X: clf.predict(X)
            path = f'/content/{name}.pkl'                  # measured locally; too large to keep on Drive
            with open(path, 'wb') as f: pickle.dump(clf, f, protocol=4)
            clf.set_params(n_jobs=1); one = lambda x: clf.predict(x)
        res['model_file_kb'] = round(os.path.getsize(path) / 1024, 1)
        if path.startswith('/content/') and not path.startswith('/content/drive'): os.remove(path)
        log(f"{name}: trained in {res['train_minutes']:.1f} min | model file {res['model_file_kb']:,.0f} KB")
        t2 = time.time(); pred = pred_fn(Xte); res['inference_seconds_test'] = round(time.time() - t2, 2)
        res['test'] = metrics(yte, pred); log_metrics(f'{name} RESULT [test]', res['test'])
        res.update(extra_merged(pred_fn, cfg, Xte, yte))
        for k in ('merged_test_CICIDS2017', 'merged_test_CIC-IoT2023', 'merged_test_CIC-IoT2023_7classes'):
            if k in res: log_metrics(f'{name} RESULT [{k}]', res[k])
        res['cpu_latency_single_stream'] = cpu_latency(one, Xte)
        log(f"{name}: CPU single-stream latency p90 {res['cpu_latency_single_stream']['p90_ms']:.3f} ms "
            f"(mean {res['cpu_latency_single_stream']['mean_ms']:.3f} ms)")
    save_atomic_json(res, result_path(name))
    log(f'{name}: saved | {sum(os.path.exists(result_path(r[0])) for r in RUNS)} of {len(RUNS)} runs finished | '
        f'session time {hm(time.time() - T0)}')
    del Xtr, ytr, Xv, yv, Xte, yte; gc.collect()
log('baselines v1: all runs finished' if all(os.path.exists(result_path(r[0])) for r in RUNS) else 'baselines v1: some runs are missing')

In [ ]:
# %% Cell 3 — summary
rows = []
for name, method, cfg in RUNS:
    if method == 'origin' or not os.path.exists(result_path(name)): continue
    r = json.load(open(result_path(name)))
    rows.append(dict(config=cfg, method=method, macro_f1=r['test']['macro_f1'], accuracy=r['test']['accuracy'],
                     false_alarm=r['test']['false_alarm_rate'], model_kb=r['model_file_kb'],
                     p90_ms=r['cpu_latency_single_stream']['p90_ms'],
                     cicids=r.get('merged_test_CICIDS2017', {}).get('macro_f1'), iot=r.get('merged_test_CIC-IoT2023', {}).get('macro_f1'),
                     iot7=r.get('merged_test_CIC-IoT2023_7classes', {}).get('macro_f1')))
print(pd.DataFrame(rows).round(4).to_string(index=False))
if os.path.exists(result_path('origin_test')):
    o = json.load(open(result_path('origin_test')))
    print('\nDataset-of-origin test:', {k: round(v, 5) for k, v in o['xgboost'].items() if isinstance(v, float)},
          '| depth-1 tree accuracy', round(o['tree_depth1']['accuracy'], 5), '| depth-2', round(o['tree_depth2']['accuracy'], 5))
save_atomic_json(dict(baselines=rows), OUT + 'summary.json')

In [ ]:
# %% Cell 4 — compact XGBoost models (merged configuration): accuracy versus size
COMPACT = [(100, 4), (50, 4), (50, 3), (20, 3)]       # (boosting rounds, maximum tree depth); no early stopping
cfg = 'config1_merged'
Xtr, ytr = load(cfg, 'train'); Xte, yte = load(cfg, 'test')
cw = class_weights(ytr); sw = np.vectorize(cw.get)(ytr).astype(np.float32)
for n_est, depth in COMPACT:
    name = f'xgboost_compact_{n_est}x{depth}_{cfg}'
    if os.path.exists(result_path(name)):
        log(f'{name}: already finished, skipping'); continue
    t1 = time.time()
    clf = xgb.XGBClassifier(n_estimators=n_est, learning_rate=0.3, max_depth=depth, tree_method='hist',
                            device='cuda' if GPU else 'cpu', objective='multi:softprob', random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr, sample_weight=sw, verbose=False)
    path = OUT + f'models/{name}.ubj'; clf.save_model(path)
    res = dict(run=name, method='xgboost_compact', config=cfg, seed=SEED, rounds=n_est, max_depth=depth, learning_rate=0.3,
               trees=int(n_est * N_CLASSES), train_minutes=round((time.time() - t1) / 60, 2),
               model_file_kb=round(os.path.getsize(path) / 1024, 1),
               nodes=int(clf.get_booster().trees_to_dataframe().shape[0]))
    pred = clf.predict(Xte); res['test'] = metrics(yte, pred)
    res.update(extra_merged(lambda X: clf.predict(X), cfg, Xte, yte))
    c = xgb.XGBClassifier(); c.load_model(path); c.set_params(device='cpu', n_jobs=1)
    res['cpu_latency_single_stream'] = cpu_latency(lambda x: c.predict(x), Xte)
    save_atomic_json(res, result_path(name))
    log(f"{name}: Macro F1 {res['test']['macro_f1']:.4f} | false alarm {res['test']['false_alarm_rate'] * 100:.2f}% | "
        f"{res['nodes']:,} tree nodes | file {res['model_file_kb']:,.0f} KB | CPU p90 {res['cpu_latency_single_stream']['p90_ms']:.3f} ms")
log('compact XGBoost models finished')

In [ ]:
# %% Cell 5 — very small XGBoost models (merged configuration), for comparison with SkyShield-IDS-S (3,358 parameters)
TINY = [(10, 3), (10, 2), (5, 3), (5, 2), (3, 2), (1, 3)]
for n_est, depth in TINY:
    name = f'xgboost_compact_{n_est}x{depth}_{cfg}'
    if os.path.exists(result_path(name)):
        log(f'{name}: already finished, skipping'); continue
    t1 = time.time()
    clf = xgb.XGBClassifier(n_estimators=n_est, learning_rate=0.3, max_depth=depth, tree_method='hist',
                            device='cuda' if GPU else 'cpu', objective='multi:softprob', random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr, sample_weight=sw, verbose=False)
    path = OUT + f'models/{name}.ubj'; clf.save_model(path)
    nodes_df = clf.get_booster().trees_to_dataframe()
    res = dict(run=name, method='xgboost_compact', config=cfg, seed=SEED, rounds=n_est, max_depth=depth, learning_rate=0.3,
               trees=int(n_est * N_CLASSES), train_minutes=round((time.time() - t1) / 60, 2),
               model_file_kb=round(os.path.getsize(path) / 1024, 1), nodes=int(nodes_df.shape[0]),
               leaves=int((nodes_df['Feature'] == 'Leaf').sum()))
    pred = clf.predict(Xte); res['test'] = metrics(yte, pred)
    res.update(extra_merged(lambda X: clf.predict(X), cfg, Xte, yte))
    c = xgb.XGBClassifier(); c.load_model(path); c.set_params(device='cpu', n_jobs=1)
    res['cpu_latency_single_stream'] = cpu_latency(lambda x: c.predict(x), Xte)
    save_atomic_json(res, result_path(name))
    log(f"{name}: Macro F1 {res['test']['macro_f1']:.4f} | false alarm {res['test']['false_alarm_rate'] * 100:.2f}% | "
        f"{res['nodes']:,} tree nodes ({res['leaves']:,} leaves) | file {res['model_file_kb']:,.0f} KB | CPU p90 {res['cpu_latency_single_stream']['p90_ms']:.3f} ms")
log('very small XGBoost models finished')

In [ ]:
# %% Cell 6 — size summary of the merged-configuration baselines (stored values = 2 x internal nodes + leaves)
import glob
rows = []
for p in sorted(glob.glob(OUT + 'results/*config1_merged*.json')):
    r = json.load(open(p)); n = r.get('nodes'); lv = r.get('leaves')
    if n is None: continue
    if lv is None:
        if r['method'] == 'random_forest':
            lv = (n + r['params']['n_estimators']) // 2          # binary trees: leaves = (nodes + trees) / 2
        else:
            c = xgb.XGBClassifier(); c.load_model(OUT + 'models/' + r['run'] + '.ubj')
            lv = int((c.get_booster().trees_to_dataframe()['Feature'] == 'Leaf').sum())
    rows.append(dict(run=r['run'], macro_f1=round(r['test']['macro_f1'], 4), far=round(100 * r['test']['false_alarm_rate'], 2),
                     nodes=n, leaves=lv, stored=2 * (n - lv) + lv, rounds=r.get('best_iteration', r.get('rounds')),
                     p90=round(r['cpu_latency_single_stream']['p90_ms'], 3), mean=round(r['cpu_latency_single_stream']['mean_ms'], 3),
                     depth=r.get('params', {}).get('max_depth', r.get('max_depth'))))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# %% Cell 7 — size of the full XGBoost model (merged configuration)
r = json.load(open(result_path('xgboost_config1_merged')))
c = xgb.XGBClassifier(); c.load_model(OUT + 'models/xgboost_config1_merged.ubj')
df = c.get_booster().trees_to_dataframe(); n = len(df); lv = int((df['Feature'] == 'Leaf').sum())
print('FULLXGB nodes', n, 'leaves', lv, 'stored', 2 * (n - lv) + lv, 'trees', df['Tree'].nunique(), 'params', r.get('params'),
      'best_iteration', r.get('best_iteration'), 'p90', r['cpu_latency_single_stream'])

In [ ]:
# %% Cell 8 — post-hoc decision offsets for the XGBoost models (same procedure as Phase 12: tuned on the validation split only)
def macro_f1_fast(y, pred, k=8):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm); p = cm.sum(0); t = cm.sum(1)
    f1 = np.where(p + t > 0, 2 * tp / np.maximum(p + t, 1), 0)
    return f1[t > 0].mean()
def tune_offsets(logp, y, grid=np.arange(-4.0, 4.01, 0.25), sweeps=3):
    b = np.zeros(logp.shape[1]); best = macro_f1_fast(y, logp.argmax(1))
    for _ in range(sweeps):
        for c in range(logp.shape[1]):
            for g in grid:
                bb = b.copy(); bb[c] = g; s = macro_f1_fast(y, (logp + bb).argmax(1))
                if s > best + 1e-6: best, b = s, bb
    return b, float(best)
Xv, yv = load(cfg, 'val')
names = ['xgboost_config1_merged'] + [f'xgboost_compact_{a}x{d}_config1_merged' for a, d in [(100,4),(50,4),(50,3),(20,3),(10,3),(5,3),(10,2),(5,2),(3,2),(1,3)]]
for name in names:
    rp = OUT + f'results/{name}_offsets.json'
    if os.path.exists(rp): log(f'{name} offsets: already done'); continue
    c = xgb.XGBClassifier(); c.load_model(OUT + f'models/{name}.ubj'); c.set_params(device='cuda' if GPU else 'cpu')
    lv = np.log(np.clip(c.predict_proba(Xv), 1e-9, 1)); lt = np.log(np.clip(c.predict_proba(Xte), 1e-9, 1))
    b, tv = tune_offsets(lv, yv); m0, m1 = metrics(yte, lt.argmax(1)), metrics(yte, (lt + b).argmax(1))
    save_atomic_json(dict(run=name, offsets=b.tolist(), val_default=float(macro_f1_fast(yv, lv.argmax(1))), val_tuned=tv, test_default=m0, test_tuned=m1), rp)
    log(f"OFFS {name}: TEST Macro F1 {m0['macro_f1']:.4f} -> {m1['macro_f1']:.4f} | false alarm {m0['false_alarm_rate']*100:.2f}% -> {m1['false_alarm_rate']*100:.2f}%")
    del c; gc.collect()
log('XGBoost offsets finished')

In [ ]:
# %% Cell 9 — post-hoc decision offsets for the full Random Forest (merged configuration), retrained with the same settings and seed
name = 'random_forest_config1_merged'; rp = OUT + f'results/{name}_offsets.json'
if os.path.exists(rp):
    log(f'{name} offsets: already done')
else:
    cw = class_weights(ytr)
    rf = RandomForestClassifier(n_estimators=100, max_samples=min(500_000, len(ytr)), min_samples_leaf=1,
                                class_weight=cw, n_jobs=-1, random_state=SEED)
    t1 = time.time(); rf.fit(Xtr, ytr); log(f'{name}: retrained in {(time.time() - t1) / 60:.1f} min')
    pv = rf.predict_proba(Xv); pt = rf.predict_proba(Xte)
    full = np.zeros((len(pv), N_CLASSES)); full[:, rf.classes_] = pv; pv = full
    full = np.zeros((len(pt), N_CLASSES)); full[:, rf.classes_] = pt; pt = full
    lv = np.log(np.clip(pv, 1e-9, 1)); lt = np.log(np.clip(pt, 1e-9, 1))
    b, tv = tune_offsets(lv, yv); m0, m1 = metrics(yte, lt.argmax(1)), metrics(yte, (lt + b).argmax(1))
    orig = json.load(open(result_path(name)))['test']['macro_f1']
    save_atomic_json(dict(run=name, offsets=b.tolist(), val_default=float(macro_f1_fast(yv, lv.argmax(1))), val_tuned=tv,
                          test_default=m0, test_tuned=m1, original_run_macro_f1=orig), rp)
    log(f"OFFS {name}: TEST Macro F1 {m0['macro_f1']:.4f} (original run {orig:.4f}) -> {m1['macro_f1']:.4f} | "
        f"false alarm {m0['false_alarm_rate']*100:.2f}% -> {m1['false_alarm_rate']*100:.2f}%")
    del rf, pv, pt, lv, lt; gc.collect()
log('Random Forest offsets finished')